## text2grammar Experiments testing


## Task3:  

Multi-Grammars Sentence → Multiple Grammars and Multiple Labels

包含多个语法的句子，然后让模型去选出包含的语法类型。


In [1]:
import pandas as pd 
import os
import pickle
from datetime import datetime, timezone
from openai import OpenAI  # pip install openai
import nltk
from nltk.tokenize import sent_tokenize
import os
from pprint import pprint
import pandas as pd 
from mt_reasoning.utils import prompts_util, clients_util 
from tqdm import tqdm
import importlib
from dotenv import load_dotenv
import random
import string

load_dotenv()

source_df = pd.read_json("data/extraction_pdf/datasets/df_samples.jsonl", lines=True)

## uv run vllm serve /home/snt/projects_lujun/base_models/gemma-2-2b-it --host 0.0.0.0 --port 1997 --max-model-len 2048 --max-num-seqs 2 --gpu-memory-utilization 0.7


In [2]:
importlib.reload(prompts_util)
importlib.reload(clients_util)

nltk.download('punkt')

## Open AI Settings
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
TEMPERATURE = float(os.environ.get("OPENAI_TEMPERATURE", "0.5"))


## VllM settings
model_vllm = os.environ.get("MODEL_VLLM", "/home/snt/projects_lujun/base_models/gemma-2-2b-it")
IP = os.environ.get("VLLM_IP", "0.0.0.0")
PORT = os.environ.get("VLLM_PORT", "1997")
server_url = f"http://{IP}:{PORT}/v1"
print (server_url)
vllm_client = OpenAI(base_url=server_url)

## Experimental Settings
sentence_list_size = 2  # Input Sentence Num to Concatenate
grammar_size = 5  # Input Grammar Descriptions Num to Evaluate
test_length = 1000  # The Tests Num We Want to evaluate

letters = list(string.ascii_uppercase)  # ['A', 'B', 'C', ..., 'Z']

time_now = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
output_dir = "/home/snt/projects_lujun/mt_reasoning/data/extraction_pdf/datasets"
output_path = os.path.join(output_dir, f"task3_{time_now}_{grammar_size}_{sentence_list_size}_{test_length}_{model_vllm.split('/')[-1]}.jsonl")


http://0.0.0.0:1997/v1


[nltk_data] Downloading package punkt to /home/snt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
for idx in tqdm(range(test_length), desc="Processing"):

    ## Randomly Sample a Row from the Source DataFrame
    rows = (
        source_df[source_df['grammar_points_descriptions'].isin(
            random.sample(list(source_df['grammar_points_descriptions'].unique()), sentence_list_size)
        )]
        .groupby('grammar_points_descriptions', group_keys=False)
        .sample(1)
        .reset_index(drop=True)
    )


    sentence_list = rows["luxembourg"].tolist()
    grammar_points = rows["grammar_points_descriptions"].drop_duplicates().tolist()
    assert len(sentence_list) == len(grammar_points), "Grammar Sentence mismatch."

    random.shuffle(sentence_list)
    paragraph = " ".join(sentence_list)
    num_grammars = sentence_list_size

    ## Grammar List Generation
    k = max(0, grammar_size - sentence_list_size)
    available = source_df.loc[~source_df['grammar_points_descriptions'].isin(grammar_points), 'grammar_points_descriptions'].drop_duplicates()
    opposite_source_sentence_list = (available.sample(n=k, replace=(k>len(available))).tolist() if k>0 and not available.empty else [])

    full_grammar_list = grammar_points + opposite_source_sentence_list
    random.shuffle(full_grammar_list)
    grammars_index = [full_grammar_list.index(g) if g in full_grammar_list else -1 for g in grammar_points]

    assert len(grammars_index) == sentence_list_size, "Grammar not found in the list."
    assert len(full_grammar_list) == grammar_size, "Sentence list size mismatch."

    option_labels = letters[:grammar_size]
    correct_grammars_letter = [option_labels[i] for i in grammars_index]
    labeled_grammars_list = [
        f"{label}. {desc}" for label, desc in zip(option_labels, full_grammar_list)
    ]
    input_dict = {
        "NUM_GRAMMARS": num_grammars,
        "PARAGRAPH":  paragraph,
        "GRAMMAR_LIST": "\n".join(labeled_grammars_list),
    }

    output_dict, input_prompt = clients_util.generate_with_calling_api(
        client=vllm_client,
        system_prompt_template_path="prompts/system/system_prompt_translation.jinja",
        input_prompt_template_path="prompts/evaluation/prompt_sentence_classification_task_3.jinja",  # Use simple, complecated one confuse the models
        input_text_dict=input_dict,
        model=model_vllm,
    )
    
    row = {
        "sentence_list": sentence_list,
        "grammar_points": grammar_points,
        "full_grammar_list": full_grammar_list,
        "option_labels": option_labels,
        "labeled_grammars_list": labeled_grammars_list,
        "input_dict": input_dict,
        "input_prompt": input_prompt,
        "task3_dict": output_dict,
        "correct_grammars_letter": correct_grammars_letter,
    }
    
    updated_row = pd.DataFrame([row])
    if not os.path.exists(output_dir):  
        os.makedirs(output_dir)
    if not os.path.exists(output_path):
        updated_row.to_json(output_path, orient="records", lines=True)
    else:
        updated_row.to_json(output_path, orient="records", lines=True, mode="a")
    
    # print(output_dict)
    # print("----------------------------------------------")
    # pprint(output_dict, indent=2, width=150, sort_dicts=False)

Processing: 100%|██████████| 1000/1000 [04:46<00:00,  3.49it/s]


In [5]:
result_df = pd.read_json("data/extraction_pdf/datasets/task3_20251003_103928_5_2_1000_gemma-2-2b-it.jsonl", lines=True)

In [ ]:
total_score = 0
for index, row in result_df.iterrows():
    correct_set = set(row['correct_grammars_letter'])
    detected_set = set(row["task3_dict"].get('grammar_selected', ''))
    score = len(detected_set & correct_set) / len(correct_set)
    total_score += score
print(f"Average Score: {total_score}/{len(result_df)} = {total_score/len(result_df):.2%}")


Average Score: 512.0/1000 = 51.20%
